In [ ]:
from google.colab import drive
drive.mount('/content/drive')


ValueError: mount failed

In [ ]:
#!ls "/content/drive/MyDrive/FB_GLE_Analysis/FB_GLE_Analysis"


# Preprocessing of gLOWCOST data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import glob
from matplotlib.patches import Patch
import itertools

In [ ]:
#reading each csv file and store them in a dictionary

def load_dataframes(folder_path, datetime_col="date"):

    dfs = {}
    for file in glob.glob(f"{folder_path}/*.csv"):
        name = file.split("/")[-1].replace(".csv", "") #extracting file name without ".csv" (these names are the keys of dictionary)
        df = pd.read_csv(file)

        df[datetime_col] = pd.to_datetime(df[datetime_col])
        df = df.set_index(datetime_col)

        dfs[name] = df

    return dfs

folder = "/content/drive/MyDrive/FB_GLE_Analysis/FB_GLE_Analysis/data"
df_dict = load_dataframes(folder, datetime_col="date")
#print(df_dict)


In [ ]:
#station metadata - altitude,latitude,station_alt(altitude of atmospheric data station), longitude,rigidity

df_dict_meta = {
    "New Mexico \n USA": {
        "df": df_dict["APO"],
        "altitude": 2790,
        "station_alt":1248,
        "latitude": 32.78,
        "longitude":-105.82,
        "rigidity":4.346
    },
    "Istanbul \n Turkey": {
        "df": df_dict["Istanbul"],
        "altitude": 40,
        "station_alt":98,
        "latitude":41.01 ,
        "longitude": 28.97,
        "rigidity":6.334
    },
    "Shinshu \n Japan": {
        "df": df_dict["Shinshu"],
        "altitude": 650,
        "station_alt":27,
        "latitude": 36.25,
        "longitude":137.98 ,
        "rigidity":10.728
    },
    "Atlanta \n USA": {
        "df": df_dict["rm415SA"],
        "altitude": 320,
        "station_alt":315,
        "latitude": 33.75,
        "longitude": -84.39,
        "rigidity":3.778
    },
    "Lund \n Sweden": {
        "df": df_dict["Lund"],
        "altitude": 59,
        "station_alt":43,
        "latitude":55.71 ,
        "longitude": 13.19,
        "rigidity":2.016
    },
    "Erzurum \n Turkey": {
        "df": df_dict["Erzurum"],
        "altitude": 3170,
        "station_alt":1758,
        "latitude":39.77 ,
        "longitude": 41.22,
        "rigidity":6.964
    },

    "Massachusetts \n USA": {
        "df": df_dict["SkyviewMS"],
        "altitude": 160,
        "station_alt":106,
        "latitude": 42.54,
        "longitude": -71.72,
        "rigidity":2.402
    },

    "California \n USA": {
        "df": df_dict["CHARA"],
        "altitude":1740 ,
        "station_alt":236,
        "latitude": 34.29,
        "longitude":-118.09 ,
        "rigidity":4.866
    },



    "Belgrade \n Serbia":{
        "df": df_dict["Serbia_Det1"],
        "altitude": 117,
        "station_alt":99,
        "latitude": 44.86,
        "longitude": 20.39,
        "rigidity":4.978
    },



    "Singapore":{
        "df": df_dict["Singapore"],
        "altitude": 15,
        "station_alt":32,
        "latitude": 1.30,
        "longitude":103.85,
        "rigidity":17.402
    },



}


In [ ]:
#sort by the altitude
sorted_sites = sorted(df_dict_meta.items(), key=lambda x: x[1]["rigidity"])
sorted_labels = [name for name, meta in sorted_sites]
#sorted_df_dict = {name: meta["df"] for name, meta in sorted_sites}
sorted_df_dict = {name: meta for name, meta in sorted_sites}

In [ ]:
#removing outliers in Erzurum station
df_erzurum_counts = sorted_df_dict['Erzurum \n Turkey']["df"]
df_erzurum_counts.loc[(df_erzurum_counts["counts12"] < 25000) | (df_erzurum_counts["counts12"]> 32000)] = np.nan
#plt.figure(figsize=(12, 6))
#plt.plot(df_erzurum_counts.index,df_erzurum_counts["counts12"])
#plt.show()

# Pressure Correction

In [ ]:
def calc_delta(df):
    df["delta_cts_12"] = np.log(df["counts12"] / df["counts12"].mean())
    df['tmpk']=(df['tmpf'] - 32) * 5/9 + 273.15
    df["delta_tmpk"]     = df["tmpk"] - df["tmpk"].mean()
    return df

def calc_pct(df):
    df['counts12_pct']=((df["counts12"]-df["counts12"].mean())/(df["counts12"].mean()))*100
    df['mslp_pct']=((df["mslp"]-df["mslp"].mean())/(df["mslp"].mean()))*100
    df['tmpf_pct']=((df["tmpf"]-df["tmpf"].mean())/(df["tmpf"].mean()))*100

def pressure_at_altitude_from_obs(mslp, tmpf, altitude,station_alt):
    T0 = (tmpf - 32) * 5 / 9 + 273.15 # Convert to Kelvin
    L  = 0.0065            # Lapse rate (K/m)
    g  = 9.80665           # m/s²
    M  = 0.0289644         # kg/mol
    R  = 8.31432           # J/(mol·K)

    T_ratio = (T0 - L * (altitude-station_alt)) / T0
    exponent = (g * M) / (R * L)

    return mslp * T_ratio ** exponent



In [ ]:
def pres_corr(df, df_baro, altitude, station_alt, start_date, end_date):

    #barometric coefficient calculation
    df_baro = df_baro.copy()

    df_baro["atm_pres"] = pressure_at_altitude_from_obs(
        df_baro["mslp"],
        df_baro["tmpf"],
        altitude,
        station_alt
    )

    df_baro["delta_pres"] = df_baro["atm_pres"] - df_baro["atm_pres"].mean()

    valid = df_baro[["delta_pres", "delta_cts_12"]].dropna()

    slope, intercept, r_value, pv, se = stats.linregress(valid["delta_pres"],valid["delta_cts_12"])

    baro = slope
    baro_error = se

    #pressure correction on full time period
    df_out = df.loc[start_date:end_date].copy()

    df_out["atm_pres"] = pressure_at_altitude_from_obs(
        df_out["mslp"],
        df_out["tmpf"],
        altitude,
        station_alt
    )

    P0 = df_out["atm_pres"].mean()
    df_out["P_minus_P0"] = df_out["atm_pres"] - P0
    df_out["pressure_correction_factor"] = np.exp(-baro * df_out["P_minus_P0"])
    df_out["Icorr_12_pres"] = df_out["counts12"] * df_out["pressure_correction_factor"]

    df_out["Icorr_12_pres"] = df_out["Icorr_12_pres"].interpolate(method="time")
    df_out["Icorr_12_pres_pct"] = ((df_out["Icorr_12_pres"] - df_out["Icorr_12_pres"].mean())/ df_out["Icorr_12_pres"].mean()) * 100

     #temp correction

    df_baro['tmpk']=(df_baro['tmpf'] - 32) * 5/9 + 273.15
    df_out['tmpk']=(df_out['tmpf'] - 32) * 5/9 + 273.15

    df_baro['delta_tmpk'] = df_baro['tmpk'] - df_baro['tmpk'].mean()

    df_baro["delta_cts_12_temp"] = (df_out["Icorr_12_pres"]-df_out["Icorr_12_pres"].mean())
    valid_temp= df_baro[["delta_tmpk", "delta_cts_12_temp"]].dropna()

    slope_t, intercept_t, r_value_t, pv_t, se_t = stats.linregress(valid_temp['delta_tmpk'], valid_temp['delta_cts_12_temp'])

    alpha=slope_t
    temp_error=se_t

    df_out["dt"] = df_out['tmpk'] - df_out['tmpk'].mean()
    df_out["Icorr_12"] = df_out["Icorr_12_pres"] - ((alpha) * df_out["dt"])

    df_out["Icorr_12"] = df_out["Icorr_12"].interpolate(method="time")

    df_out["Icorr_12_pct"] = ((df_out["Icorr_12"] - df_out["Icorr_12"].mean())/ df_out["Icorr_12"].mean()) * 100


    return df_out, df_baro, baro, alpha, baro_error, temp_error

In [ ]:
baro_values = {}

for name, meta in sorted_df_dict.items():
    df = meta["df"]
    altitude = meta["altitude"]
    station_alt = meta["station_alt"]

    # This period is only for calculating baro
    df_filtered = df.loc["2025-09-19":"2025-11-01"].copy()

    calc_delta(df_filtered)
    calc_pct(df)

    df_corr, df_baro, baro, alpha, baro_error, temp_error= pres_corr(
        df,
        df_filtered,
        altitude,
        station_alt,
        "2025-09-19",
        "2025-11-19"
    )
    meta["df"] = df_corr
    baro_values[name] = {
      "baro": baro,
      "baro_error": baro_error,
      "alpha": alpha,
      "alpha_error": temp_error
  }

    #print(f"{name}: baro = {baro:.6f} \u00b1 {se:.6f}")
    #df_corr.to_csv(f"/content/drive/MyDrive/FB_GLE_Analysis/FB_GLE_Analysis/pressure_corrected_data/{name}_press_corr.csv")

In [ ]:
#barometric coefficient values

baro_df = pd.DataFrame.from_dict(
    baro_values,
    orient="index",
    #columns=["baro_coeff", "uncertainty"]
)
baro_df = baro_df.round(5)

#baro_df.to_csv("/content/drive/MyDrive/FB_GLE_Analysis/FB_GLE_Analysis/baro_values.csv")
baro_df

# FB/GLE Visualization for gLOWCOST

In [ ]:
def plot_panel_highlighted(
    df_meta_dict,             #metadata dictionary
    cols,                     #columns to be plotted
    start=None,               #start date for plotting
    end=None,                 #end date for plotting
    figsize=None,             #figure size
    sharex=True,              #common x axis
    highlight_centers=None,   #time of events recorded (IPS & GLE)
    highlight_window_hours=12,#highlight window size to one side of center
    title="None",             #title for the plot
    highlight_colors=None,    #colors for highlight windows
    highlight_labels=None,    #labels for highlight windows
    bbox_loc=(0.5, 0.98),     # legend box placement
):
    keys = list(df_meta_dict.keys())
    n = len(keys)

    if figsize is None:
        figsize = (12, 3 * n)

    fig, axes = plt.subplots(n, 1, figsize=figsize, sharex=sharex)

    # computing highlight windows
    highlight_windows = []
    if highlight_centers is not None:
        for t in highlight_centers:
            center = pd.to_datetime(t)
            half = pd.Timedelta(hours=highlight_window_hours)
            highlight_windows.append((center - half, center + half))

    # building higlight colors list
    if highlight_windows:
        if highlight_colors is None:
            # use default matplotlib color cycle
            default_cycle = plt.rcParams["axes.prop_cycle"].by_key().get("color", ["C0"])
            colors = list(itertools.islice(itertools.cycle(default_cycle),len(highlight_windows)))
        else:
            colors = highlight_colors
    else:
        colors = []

    # building labels list for highlight windows
    if highlight_windows:
        if highlight_labels is None:
            highlight_labels = [None] * len(highlight_windows)
    else:
        highlight_labels = []

    # to collect band legend handles globally
    global_band_handles = []

    for ax, key in zip(axes, keys):
        meta = df_meta_dict[key]
        df = meta["df"]

        # time window
        if start or end:
            df = df.loc[start:end]

        # overlay columns
        for col in cols:
            if col in df.columns:
                ax.plot(df.index, df[col],label="Pressure & Temperature corrected\n hourly counts",linewidth=1)

        # highlight windows and append legend label
        if highlight_windows:
            for (h_start, h_end), color, label in zip(highlight_windows, colors, highlight_labels):
                span = ax.axvspan(h_start, h_end, alpha=0.3, color=color)

                # only add a legend handle if we have a label (to avoid having labels for dotted lines)
                if label is not None:
                    # only add each label once (avoid duplicates across axes)
                    if not any(h.get_label() == label for h in global_band_handles):
                        global_band_handles.append(Patch(facecolor=color, alpha=0.3, label=label))

        ax.set_ylabel(key, fontsize=12, fontweight="bold")
        ax.grid(True, alpha=0.3)

        # drawing vertical dotted line at highlight centers
        if highlight_centers is not None:
            for t in highlight_centers:
                center = pd.to_datetime(t)
                ax.axvline(center, color="black", linestyle="--", linewidth=1)


    axes[-1].set_xlabel("Timestamp (UTC)",fontsize=12, fontweight="bold")
    plt.setp(axes[-1].get_xticklabels(), rotation=30, ha="right")

    # Creating GLOBAL LEGEND

    # take line handles/labels from first axis
    line_handles, line_labels = axes[0].get_legend_handles_labels()

    handles = line_handles + global_band_handles
    labels  = line_labels + [h.get_label() for h in global_band_handles]

    if handles:
        fig.legend(
            handles,
            labels,
            loc="upper center",
            ncol=1,
            fontsize=10,
            frameon=True,
            bbox_to_anchor=bbox_loc
        )

    fig.tight_layout(rect=[0, 0, 1, 1])

    plt.savefig(f"/content/drive/MyDrive/FB_GLE_Analysis/FB_GLE_Analysis/images_paper/{title}.png")
    plt.show()


In [ ]:


plot_panel_highlighted(
    sorted_df_dict,
    cols=["Icorr_12"],
    start="2025-11-01",
    end="2025-11-18",
    figsize=(12, 15),
    sharex=True,
    highlight_centers=[
        "2025-11-07 22:00",
        "2025-11-11 10:00",
        "2025-11-11 22:00",
    ],
    highlight_window_hours=12,
    title="Coincident count with FBD and GLE days - gLOWCOST Network",
    highlight_colors=["tab:red", "tab:green", "tab:blue"],
    highlight_labels=[
        r"IPS: 2025-11-07 22:00 $\pm$ 12 h",
        r"GLE: 2025-11-11 10:00 $\pm$ 12 h",
        r"IPS: 2025-11-11 22:00 $\pm$ 12 h",
    ],
    bbox_loc=(0.85, 0.94)
)

# NM Data

In [ ]:
def NM_calc_pct(df):
    df['Oulu_pct']=((df["Oulu_counts"]-df["Oulu_counts"].mean())/(df["Oulu_counts"].mean()))*100
    df['Newark_pct']=((df["Newark_counts"]-df["Newark_counts"].mean())/(df["Newark_counts"].mean()))*100
    df['Mexc_pct']=((df["Mexc_counts"]-df["Mexc_counts"].mean())/(df["Mexc_counts"].mean()))*100
    df['SOPO_pct']=((df["SOPO_counts"]-df["SOPO_counts"].mean())/(df["SOPO_counts"].mean()))*100

In [ ]:
df_NM = pd.read_csv("/content/drive/MyDrive/FB_GLE_Analysis/FB_GLE_Analysis/NM/nmdb_data_hourly.csv",index_col="datetime",parse_dates=["datetime"])
NM_calc_pct(df_NM)
#df_NM


In [ ]:
#metadata for NM stations

df_NM_meta = {
    "SOPO NM": {
        "df": df_NM[["SOPO_counts"]].rename(columns={"SOPO_counts": "NM_counts"}),
        "altitude": 2820,
        "latitude": -90.00,
        "longitude": None,
        "rigidity": 0.1,
    },

    "Oulu NM": {
        "df": df_NM[["Oulu_counts"]].rename(columns={"Oulu_counts": "NM_counts"}),
        "altitude": 15,
        "latitude": 65.05,
        "longitude": 25.47,
        "rigidity": 0.8,
    },

    "Mexico NM": {
        "df": df_NM[["Mexc_counts"]].rename(columns={"Mexc_counts": "NM_counts"}),
        "altitude": 2274,
        "latitude": 19.33,
        "longitude": 260.82,
        "rigidity": 8.2,
    },
}


# Barbashina(2009) method for FB amplitude calculation

In [ ]:
#function for detrending time series segments before and after the FBD phase

def linear_detrend_segment(ts):

    ts = ts.dropna()
    if ts.empty:
        return ts

    x = (ts.index - ts.index[0]).total_seconds() / 3600.0  # x = time in hours from start (x= 0,1,2.....)
    y = ts.values                                          # y = press_corr_pct values for each x points

    a, b = np.polyfit(x, y, 1) # Linear fit: y ≈ a*x + b for the segment of interest

    trend = a * x + b #trend
    y_detr = y - trend + y.mean()   # remove trend but keep mean

    return pd.Series(y_detr, index=ts.index)

In [ ]:
#function for detrending pre and post segment used for visualizing part
# s = time series , h_start = starting time of the descrease , h_end = ending time of the decrease

def detrend_pre_post_around_window(s, h_start, h_end):

    out = pd.Series(index=s.index, dtype=float)

    pre  = s.loc[:h_start] #pre segment of the time series
    post = s.loc[h_end:]   #post segment of the time series

    pre_d  = linear_detrend_segment(pre) #detrending the pre time series
    post_d = linear_detrend_segment(post) #detrending the post time series


    out.loc[pre_d.index] = pre_d
    out.loc[post_d.index] = post_d
    out.loc[h_start:h_end] = np.nan

    return out


In [ ]:
def compute_before_means(
    before_seg,
    max_days = 3,
    min_points = 10,
):

    before_means = {}


    s = before_seg.dropna()
    if s.empty:
        return before_means

    one_day = pd.Timedelta(days=1) #defining min timedleta
    full_max = s.index.max() # maximum datetime index in the chosen segment / FBD starting point


    for i in range(1, max_days + 1):  # i = size of the detrended window in days (1, 2, 3)

        start_i = full_max - i * one_day # Take the last i days of the pre-FD segment
        seg_i = s.loc[s.index >= start_i].dropna() #time series data for the chosen i

        if len(seg_i) < min_points:
            continue

        # Detrending the segment of chosen window size
        seg_i_detr = linear_detrend_segment(seg_i)

        if seg_i_detr.empty:
            continue

        det_max = seg_i_detr.index.max()


        for j in range(1, i + 1): # j = size of the final averaging window in days (1..i)


            start_j = det_max - j * one_day # Take the last j days of the detrended segment
            sub = seg_i_detr.loc[seg_i_detr.index >= start_j].dropna() #time series data for the chosen j

            if len(sub) < min_points:
                continue

            before_means[(i, j)] = float(sub.mean()) #calculating the mean

    return before_means


In [ ]:
def compute_after_means(
    after_seg,
    max_days = 3,
    min_points = 10,
):

    after_means = {}


    s = after_seg.dropna()
    if s.empty:
        return after_means

    one_day = pd.Timedelta(days=1) #defining min timedleta
    full_max = s.index.min() # min datetime index in the chosen segment / FBD starting point


    for i in range(1, max_days + 1):  # i = size of the detrended window in days (1, 2, 3)

        end_i = full_max + i * one_day # Take the first i days of the pre-FD segment
        seg_i = s.loc[s.index <= end_i].dropna() #time series data for the chosen i

        if len(seg_i) < min_points:
            continue

        # Detrending the segment of chosen window size
        seg_i_detr = linear_detrend_segment(seg_i)

        if seg_i_detr.empty:
            continue

        det_max = seg_i_detr.index.min()


        for j in range(1, i + 1): # j = size of the final averaging window in days (1..i)


            end_j = det_max + j * one_day # Take the first j days of the detrended segment
            sub = seg_i_detr.loc[seg_i_detr.index <= end_j].dropna() #time series data for the chosen j

            if len(sub) < min_points:
                continue

            after_means[(i, j)] = float(sub.mean()) #calculating the mean

    return after_means


In [ ]:
def compute_fd_amplitude(
    df,
    col,
    t1, #FB starting time
    t2, #FB ending time
    max_days_pre=3, # maximum number of days pre FB
    max_days_post=3, # maximum number of days poest FB
    min_points=10,
):


    t1 = pd.to_datetime(t1)
    t2 = pd.to_datetime(t2)

    one_day = pd.Timedelta(days=1)

    before_mask = (df.index < t1) & (df.index >= t1 - max_days_pre * one_day)  # pre FB max segment boolean mask
    after_mask  = (df.index > t2) & (df.index <= t2 + max_days_post * one_day) # post FB max segment boolean mask

    before_seg = df.loc[before_mask, col].dropna() #pre FB segment data
    after_seg  = df.loc[after_mask,  col].dropna() #post FB segment data

    #check if the pre and post segments are empty
    if before_seg.empty:
        raise ValueError("No data available in the pre-FD interval.")
    if after_seg.empty:
        raise ValueError("No data available in the post-FD interval.")

    # computing pre and post means
    before_means = compute_before_means(
        before_seg,
        max_days=max_days_pre,
        min_points=min_points,
    )
    after_means = compute_after_means(
        after_seg,
        max_days=max_days_post,
        min_points=min_points,
    )

    #check if means were calculated
    if not before_means:
        raise RuntimeError("Failed to compute any 'before' means.")
    if not after_means:
        raise RuntimeError("Failed to compute any 'after' means.")

    # calculating FB depth for each of the mean values
    A_values = []

    for (i, j), Ib in before_means.items():
        for (k, m), Ir in after_means.items():
            if Ib == 0 or np.isnan(Ib) or np.isnan(Ir):
                continue
            A = (Ib - Ir) / Ib *100.0   # FD amplitude in percent
            A_values.append(A)

    if not A_values:
        raise RuntimeError("No valid FD amplitudes could be computed.")

    A_values = np.array(A_values, dtype=float)
    A_FD = float(A_values.mean())
    sigma_A = float(A_values.std(ddof=1)) if len(A_values) > 1 else 0.0

    return {
        "A_FD": A_FD,
        "sigma_A": sigma_A,
        "A_values": A_values,
        "before_means": before_means,
        "after_means": after_means,
    }


# Forbush 1 --->  Amplitude

In [ ]:


def compute_fd_for_all_FB1(
    df_dict_meta,
    col,
    t1,
    t2,
    max_days_pre=3,
    max_days_post=3,
    min_points=10,
    mavg_hours=1,
):

    results = {}
    rows = []

    for name, meta in df_dict_meta.items():
        if isinstance(meta, dict) and "df" in meta:
            df = meta["df"]
        else:
            df = meta

        try:
            df[col]=df[col].rolling(window=mavg_hours).mean()
            max_time, min_time = t1,t2 #max_min_time(
                #df=df,
                #col=col,
                #t1=t1,
                #t2=t2,
                #unit="H",
            #)
            res = compute_fd_amplitude(
                df=df,
                col=col,
                t1=max_time,
                t2=min_time,
                max_days_pre=max_days_pre,
                max_days_post=max_days_post,
                min_points=min_points,
            )
            results[name] = res

            rows.append({
                "detector": name,
                "Amplitude (A)": np.round(res["A_FD"],2),
                "sigma_A": np.round(res["sigma_A"],2),
                "n_amps": len(res["A_values"]),  # how many (i,j,k,m) combos worked
            })

        except Exception as e:
            # If something fails for one detector, log it and continue
            print(f"[WARN] FD computation failed for {name}: {e}")

    if rows:
        summary_df = pd.DataFrame(rows).set_index("detector")
    else:
        summary_df = pd.DataFrame(columns=["Amplitude (A)", "sigma_A", "n_amps"])

    return summary_df, results


In [ ]:
summary_df_MD_FB1, fd_results_MD_FB1 = compute_fd_for_all_FB1(
    sorted_df_dict,
    col="Icorr_12",
    t1="2025-11-06 22:00:00+00:00",
    t2="2025-11-08 04:00:00+00:00",
    max_days_pre=2,
    max_days_post=2,
    min_points=10,
    mavg_hours=1,
)

#summary_df.to_csv("/content/drive/MyDrive/FB_GLE_Analysis/FB_GLE_Analysis/FB_amplitude.csv")

print(summary_df_MD_FB1)


In [ ]:
summary_df_NM_FB1, fd_results_NM_FB1= compute_fd_for_all_FB1(
    df_NM_meta,
    col="NM_counts",
    t1="2025-11-06 22:00:00+00:00",
    t2="2025-11-08 04:00:00+00:00",
    max_days_pre=2,
    max_days_post=2,
    min_points=10,
)

#summary_df.to_csv("/content/drive/MyDrive/FB_GLE_Analysis/FB_GLE_Analysis/FB_amplitude.csv")

print(summary_df_NM_FB1)

In [ ]:
summary_df_FB1= pd.concat([summary_df_MD_FB1,summary_df_NM_FB1])
summary_df_FB1

#summary_df.to_csv("/content/drive/MyDrive/FB_GLE_Analysis/FB_GLE_Analysis/FB_amplitude.csv")

# Forbush 2 ---> Amplitude

In [ ]:
summary_df_MD_FB2, fd_results_MD_FB2 = compute_fd_for_all_FB1(
    sorted_df_dict,
    col="Icorr_12",
    t1="2025-11-11 08:00:00+00:00",
    t2="2025-11-12 08:00:00+00:00",
    max_days_pre=2,
    max_days_post=2,
    min_points=10,
    mavg_hours=1,
)

#summary_df.to_csv("/content/drive/MyDrive/FB_GLE_Analysis/FB_GLE_Analysis/FB_amplitude.csv")

print(summary_df_MD_FB2)

In [ ]:
summary_df_NM_FB2, fd_results_NM_FB2= compute_fd_for_all_FB1(
    df_NM_meta,
    col="NM_counts",
    t1="2025-11-11 08:00:00+00:00",
    t2="2025-11-12 04:00:00+00:00",
    max_days_pre=2,
    max_days_post=2,
    min_points=10,
)

#summary_df.to_csv("/content/drive/MyDrive/FB_GLE_Analysis/FB_GLE_Analysis/FB_amplitude.csv")

print(summary_df_NM_FB2)

In [ ]:
summary_df_FB2= pd.concat([summary_df_MD_FB2,summary_df_NM_FB2])
summary_df_FB2

# FB Amplitude visualization

In [ ]:
summary_df= summary_df_FB1.merge(summary_df_FB2, left_index=True, right_index=True)
summary_df.drop(columns=["n_amps_x", "n_amps_y"], inplace=True)
summary_df.rename(columns={"Amplitude (A)_x": "A_FB1","Amplitude (A)_y": "A_FB2","sigma_A_x":"sigma_FB1","sigma_A_y":"sigma_FB2"}, inplace=True)
print(summary_df)

summary_df.to_csv("/content/drive/MyDrive/FB_GLE_Analysis/FB_GLE_Analysis/FB1_FB2_amplitude.csv")

In [ ]:
summary_df_MD = summary_df_MD_FB1.merge(summary_df_MD_FB2, left_index=True, right_index=True)
summary_df_MD.drop(columns=["n_amps_x", "n_amps_y"], inplace=True)
summary_df_MD.rename(columns={"Amplitude (A)_x": "A_FB1","Amplitude (A)_y": "A_FB2","sigma_A_x":"sigma_FB1","sigma_A_y":"sigma_FB2"}, inplace=True)
summary_df_MD
